# Data Preparation for NFT Market Regimes Dataset

This notebook documents the setup and table-construction flow for rebuilding the BigQuery-side prepared dataset from the public Zenodo release.

Covered scope:

- upload public source files to GCS,
- load `nft_trading_base`, `nft_metadata_base`, and `usd_eth_base`,
- create `nft_trading_usd`, `nft_trading_usd_prefilter`, and `nft_trading_usd_filtered`,
- run simple validation queries.

Schema assumptions used below are aligned with the Zenodo dataset README:

- `nft_trading_base.timestamp` is a **timestamp** column with time information,
- `nft_trading_base.price_eth` and `nft_trading_base.fee_eth` are numeric ETH-denominated values,
- the Etherscan CSV provides daily ETH/USD values with fields `Date(UTC)`, `UnixTimeStamp`, and `Value`.


## 0. Before you start

Authenticate on your local machine before running GCP-related cells.

```bash
gcloud auth login
gcloud config set project your-gcp-project
gcloud auth application-default login
```

Confirm the active project:

```bash
gcloud config get-value project
```


In [ ]:
from pathlib import Path
import os
import pandas as pd
from google.cloud import bigquery

# --- Configuration ---
PROJECT_ID = "your-gcp-project"
DATASET_ID = "your_dataset"
LOCATION = "asia-northeast1"
BUCKET = "your-bucket"
GCS_PREFIX = "nft_market_regimes/raw"

LOCAL_RAW_DIR = Path("data/raw")
LOCAL_EXTRACTED_DIR = Path("data/extracted")

BQ_DATASET = f"{PROJECT_ID}.{DATASET_ID}"
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

print(PROJECT_ID)
print(BQ_DATASET)


## 1. Download public source files

Run these commands in a terminal.

```bash
mkdir -p data/raw data/extracted
curl -L -o data/raw/nft_trading_base.tar.gz "https://zenodo.org/records/19111864/files/nft_trading_base.tar.gz?download=1"
curl -L -o data/raw/nft_metadata_base.parquet.gz "https://zenodo.org/records/19111864/files/nft_metadata_base.parquet.gz?download=1"
tar -xzf data/raw/nft_trading_base.tar.gz -C data/extracted/
```

Separately, download ETH/USD daily price data as `data/raw/etherprice.csv`.


In [ ]:
# Inspect local files
for p in sorted(LOCAL_RAW_DIR.glob('*')):
    print('RAW   ', p)

for p in sorted(LOCAL_EXTRACTED_DIR.glob('*')):
    print('EXTRACT', p)


## 2. Upload source files to GCS

Use either `gcloud storage cp` in a terminal or Python client operations. Terminal examples:

```bash
gcloud storage cp data/raw/nft_metadata_base.parquet.gz gs://your-bucket/nft_market_regimes/raw/
gcloud storage cp data/raw/etherprice.csv gs://your-bucket/nft_market_regimes/raw/
gcloud storage cp --recursive data/extracted/nft_trading_base gs://your-bucket/nft_market_regimes/raw/nft_trading_base/
```


## 3. Create dataset if needed


In [ ]:
dataset = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset.location = LOCATION
client.create_dataset(dataset, exists_ok=True)
print(f"Dataset ready: {PROJECT_ID}.{DATASET_ID}")


## 4. Load `nft_metadata_base` from parquet

The Zenodo release provides `nft_metadata_base.parquet.gz`. In practice, it is usually simplest to decompress it locally first and then upload the `.parquet` file to GCS.

Example terminal step:

```bash
gunzip -c data/raw/nft_metadata_base.parquet.gz > data/raw/nft_metadata_base.parquet
gcloud storage cp data/raw/nft_metadata_base.parquet gs://your-bucket/nft_market_regimes/raw/
```


In [ ]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

uri = f"gs://{BUCKET}/{GCS_PREFIX}/nft_metadata_base.parquet"
table_id = f"{BQ_DATASET}.nft_metadata_base"

load_job = client.load_table_from_uri(uri, table_id, job_config=job_config)
load_job.result()
print(f"Loaded: {table_id}")


## 5. Load `nft_trading_base` from parquet parts

If the extracted archive contains many parquet files, use a wildcard URI.


In [ ]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

uri = f"gs://{BUCKET}/{GCS_PREFIX}/nft_trading_base/*.parquet"  # update if needed
table_id = f"{BQ_DATASET}.nft_trading_base"

load_job = client.load_table_from_uri(uri, table_id, job_config=job_config)
load_job.result()
print(f"Loaded: {table_id}")


## 6. Load `usd_eth_base` from CSV

The Zenodo dataset README states that the ETH/USD CSV downloaded from Etherscan contains the following fields:

- `Date(UTC)`
- `UnixTimeStamp`
- `Value`

The cells below first preview the CSV locally and then load it into BigQuery.


In [ ]:
csv_preview = pd.read_csv(LOCAL_RAW_DIR / 'etherprice.csv')
csv_preview.head()


In [ ]:
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

uri = f"gs://{BUCKET}/{GCS_PREFIX}/etherprice.csv"
table_id = f"{BQ_DATASET}.usd_eth_base"

load_job = client.load_table_from_uri(uri, table_id, job_config=job_config)
load_job.result()
print(f"Loaded: {table_id}")


## 7. Inspect schemas

Check actual column names before finalizing SQL.


In [ ]:
for table_name in ['nft_trading_base', 'nft_metadata_base', 'usd_eth_base']:
    table = client.get_table(f"{BQ_DATASET}.{table_name}")
    print(f'\n=== {table_name} ===')
    for field in table.schema:
        print(field.name, field.field_type)


## 8. Create `nft_trading_usd`

This query uses the confirmed source-column names:

- `nft_trading_base.timestamp` as the transaction timestamp,
- `nft_trading_base.price_eth` as the ETH-denominated trade value,
- `nft_trading_base.fee_eth` as the ETH-denominated fee value,
- `usd_eth_base.Date(UTC)` and `usd_eth_base.Value` from the Etherscan CSV.

Because `timestamp` contains time information while the ETH/USD file is daily, the join uses `DATE(t.timestamp)`.


In [ ]:
sql_nft_trading_usd = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET}.nft_trading_usd` AS
WITH fx AS (
  SELECT
    PARSE_DATE('%Y-%m-%d', `Date(UTC)`) AS price_date,
    CAST(`Value` AS FLOAT64) AS eth_usd_rate
  FROM `{BQ_DATASET}.usd_eth_base`
)
SELECT
  t.*,
  fx.eth_usd_rate,
  t.price_eth * fx.eth_usd_rate AS price_usd,
  t.fee_eth * fx.eth_usd_rate AS fee_usd
FROM `{BQ_DATASET}.nft_trading_base` AS t
LEFT JOIN fx
  ON DATE(t.timestamp) = fx.price_date
"""

query_job = client.query(sql_nft_trading_usd)
query_job.result()
print('Created nft_trading_usd')


## 9. Create `nft_trading_usd_prefilter`

This table applies only a minimal prefilter that is broadly reusable:

- keep rows with a matched ETH/USD rate,
- keep rows with non-null and positive USD prices.

You can extend this step later if your project requires additional intermediate exclusions.


In [ ]:
sql_prefilter = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET}.nft_trading_usd_prefilter` AS
SELECT *
FROM `{BQ_DATASET}.nft_trading_usd`
WHERE eth_usd_rate IS NOT NULL
  AND price_usd IS NOT NULL
  AND price_usd > 0
"""

query_job = client.query(sql_prefilter)
query_job.result()
print('Created nft_trading_usd_prefilter')


## 10. Create `nft_trading_usd_filtered`

The final filtering logic depends on the analytical rules used in your project. Since those rules are not fully specified in the public dataset README, this notebook keeps the final step as a clear placeholder.

Replace the `WHERE TRUE` clause below with your project-specific filtering conditions.


In [ ]:
sql_filtered = f"""
CREATE OR REPLACE TABLE `{BQ_DATASET}.nft_trading_usd_filtered` AS
SELECT *
FROM `{BQ_DATASET}.nft_trading_usd_prefilter`
WHERE TRUE
"""

query_job = client.query(sql_filtered)
query_job.result()
print('Created nft_trading_usd_filtered')


## 11. Validation queries


In [ ]:
validation_sql = f"""
SELECT 'nft_trading_base' AS table_name, COUNT(*) AS n FROM `{BQ_DATASET}.nft_trading_base`
UNION ALL
SELECT 'nft_metadata_base', COUNT(*) FROM `{BQ_DATASET}.nft_metadata_base`
UNION ALL
SELECT 'usd_eth_base', COUNT(*) FROM `{BQ_DATASET}.usd_eth_base`
UNION ALL
SELECT 'nft_trading_usd', COUNT(*) FROM `{BQ_DATASET}.nft_trading_usd`
UNION ALL
SELECT 'nft_trading_usd_prefilter', COUNT(*) FROM `{BQ_DATASET}.nft_trading_usd_prefilter`
UNION ALL
SELECT 'nft_trading_usd_filtered', COUNT(*) FROM `{BQ_DATASET}.nft_trading_usd_filtered`
"""

client.query(validation_sql).to_dataframe()
